## SQL Slip Pull

In [5]:
import numpy as np
import pandas as pd

In [ ]:
import pyodbc
import pandas as pd
import textwrap

# ── Connection parameters ──────────────────────────────────────────────────────
driver   = "ODBC Driver 18 for SQL Server"
server   = "syn-digital-zuse2-prod.sql.azuresynapse.net"   # Azure Synapse
database = "DedicatedSQLPool"

connection_string = f"""
Driver={{{driver}}};
Server={server};
Database={database};
Authentication=ActiveDirectoryInteractive;
Encrypt=yes;
TrustServerCertificate=no;
"""

# ── Run the query and grab the results into df_main ───────────────────────────
try:
    conn   = pyodbc.connect(connection_string, autocommit=True)
    cursor = conn.cursor()

    main_query_sql = textwrap.dedent("""
     SELECT 
            *,
        CASE 
            WHEN Raw_Stage IN ('Closed Deferred','Closed Lost') THEN 'Closed'
            WHEN Raw_Stage IN ('6 - Closed/Pending','Closed Won','Stage 5 - Closed Won') THEN 'Closed Won'
            WHEN Raw_Stage IN (
                     'Closed - Duplicate',
                     'Stage 6 - Closed - Admin',
                     'Stage 7 - Churned',
                     'Opportunity Rejected',
                     '0 - First Interaction'
                ) THEN 'Other'
                ELSE 'Open'
            END AS Stage
        FROM [rep].[trf_opp_daily_snapshot_new]
        Where IsQuarterWeekStartDate = 1
        AND Stage_Pipe_Category <> 'Meeting'
       AND Stage_Pipe_Category IS NOT NULL 
       AND snapshot_date >= '2026-01-01'
	   AND snapshot_date <= '2026-12-31'
       AND CloseDate >= QuarterStartDate
       AND CloseDate <= Next2QtrEndDate
      AND Raw_Stage NOT IN (
      'Churned',
     'Opportunity Rejected',
     'Stage 6 - Closed - Admin',
     'Stage 7 - Churned')
    """)
    
    cursor.execute(main_query_sql)
    rows       = cursor.fetchall()
    columns    = [col[0] for col in cursor.description]
    df_main    = pd.DataFrame.from_records(rows, columns=columns)

    # Show a preview of df_main
    df_main.head()

    # Optionally save to CSV
    # df_main.to_csv("output.csv", index=False)

except pyodbc.Error as ex:
    print(f"Database error: {ex}")
except Exception as e:
    print(f"General error: {e}")
finally:
    try:
        cursor.close()
        conn.close()
    except:
        pass

In [3]:
df_main.head()

,Opp_Id,Bookings_Team_static,BTS_lower,Account_Id,opportunity_source,Opportunity_OwnerId,Opportunity_Owner,Raw_Stage,Manager_Forecast_Category,Opp_Type,...,Stage_5_start_date,Next_Steps_Last_Updated_date,snaplogic_extract_date,age_in_days_since_s1,S1_Age,NextStep_Age,maxstagedate,Stage_Age,age_in_days (bucket),Stage
0,006Po00000WQYjaIAH,EMEA Core UKI,emea core uki,0018c000030lGsiAAE,Sales Sourced,0058c00000EQjfhAAD,Emma Colleran,2 - Qualification Status,Pipeline,Upsell,...,None,2025-01-24,2025-05-07 20:09:38.162,320.0,79.0,79.0,2025-04-10,3.0,180 - 360 Days,Open
1,006Po00000sarmfIAA,EMEA Core France Emerging,emea core france emerging,001Po000019EWqTIAW,Partner Sourced,005Po00000QlYKDIA3,Sara Capriolo,1 - Discovery,Pipeline,Expansion,...,None,2025-10-22,2025-12-18 23:16:23.819,148.0,60.0,60.0,2025-10-22,60.0,90 - 180 Days,Open
2,006Po00000N6YhiIAF,EMEA Core Alps CEE,emea core alps cee,0018c000030kkACAAY,Partner Sourced,0058c00000DXv9UAAT,Christoph Heimerl,5 - Negotiation / Business Procurement,Likely,Expansion,...,2024-12-19,2025-01-24,2025-01-31 20:07:34.489,131.0,109.0,9.0,2024-12-19,45.0,90 - 180 Days,Open
3,006Po00000OWVJ1IAP,EMEA DevOps,emea devops,0018c000030kpAQAAY,Sales Sourced,005Po00000J4yVhIAJ,Roman Pesin,1 - Discovery,Upside,Upsell,...,None,2025-01-24,2025-01-31 10:00:21.995,180.0,93.0,9.0,2024-11-01,93.0,90 - 180 Days,Open
4,0068c00000zFyqxAAC,APAC ANZ,apac anz,0018c000034OWYBAA4,Partner Sourced,0058c00000DXvP9AAL,Robert Yue,1 - Discovery,Pipeline,New Business,...,None,2024-09-30,2025-01-31 10:00:21.995,200.0,125.0,125.0,2024-09-30,125.0,180 - 360 Days,Open


## Look at snapshot date as of today and how much pipeline slipped from today to Q2 to get next quarter Forecast

In [11]:
print(df_main["snapshot_date"].dtype)
print(df_main["snapshot_date"].nunique)
print(df_main["snapshot_date"].sort_values().unique()[:20])

object
<bound method IndexOpsMixin.nunique of 0         2025-04-13
1         2025-12-21
2         2025-02-02
3         2025-02-02
4         2025-02-02
             ...    
403567    2025-07-13
403568    2025-07-13
403569    2025-07-13
403570    2025-01-26
403571    2025-01-26
Name: snapshot_date, Length: 403572, dtype: object>
[datetime.date(2025, 1, 1) datetime.date(2025, 1, 5)
 datetime.date(2025, 1, 12) datetime.date(2025, 1, 19)
 datetime.date(2025, 1, 26) datetime.date(2025, 2, 2)
 datetime.date(2025, 2, 9) datetime.date(2025, 2, 16)
 datetime.date(2025, 2, 23) datetime.date(2025, 3, 2)
 datetime.date(2025, 3, 9) datetime.date(2025, 3, 16)
 datetime.date(2025, 3, 23) datetime.date(2025, 3, 31)
 datetime.date(2025, 4, 1) datetime.date(2025, 4, 6)
 datetime.date(2025, 4, 13) datetime.date(2025, 4, 20)
 datetime.date(2025, 4, 27) datetime.date(2025, 5, 4)]


In [21]:
import datetime
df = df_main[df_main["snapshot_date"] == datetime.date(2025, 3, 16)]
## Filter on values that are Open Stage Set to Close In This Month
df = df[df["Stage"] == "Open"]
df = df[(df["CloseDate"] > datetime.date(2026, 3, 19)) & (df["CloseDate"] <= datetime.date(2026, 3, 31))]

In [25]:
## Get list of Unique Opp Id in this snapshot 
opp_ids = df["Opp_Id"].unique()
print(f"Opps closing 3/19-3/31 as of 3/16 snapshot: {len(opp_ids)}")

Opps closing 3/19-3/31 as of 3/16 snapshot: 23


In [26]:
## total NACV for these opps
print(f'Total NACV: {df["Total_NACV"].sum():,.2f}')

Total NACV: 2,257,014.61


In [ ]:
## pull those same opps from the Q3 snapshot (4/1)
df_q2 = df_main[
    (df_main["snapshot_date"] == datetime.date(2025, 4, 1)) &
    (df_main["Opp_Id"].isin(opp_ids))
]
print(f"Opps found in 4/1 snapshot: {df_q2['Opp_Id'].nunique()}")

Opps found in 4/1 snapshot: 22


In [29]:
## which ones slipped to Q2 (close date moved past 3/31)
df_q2_slipped = df_q2[df_q2["CloseDate"] > datetime.date(2025, 3, 31)]
print(f'Slipped to Q2: {df_q2_slipped["Opp_Id"].nunique()} opps')
print(f'Slipped NACV: {df_q2_slipped["Total_NACV"].sum():,.2f}')

Slipped to Q2: 22 opps
Slipped NACV: 1,805,606.63


In [30]:
## % of NACV that slipped to Q2
pct_slipped = df_q2_slipped["Total_NACV"].sum() / df["Total_NACV"].sum() * 100
print(f'Slipped: {pct_slipped:,.1f}% of NACV')

Slipped: 80.0% of NACV


## Next Snap Shot